# Prototype 1 — Single-Item Segmentation + Tagging

Validates the single-item pipeline: load → categorize → background removal → data card.

Inputs: one of `inputs/` photos (or any `~/.closet/images/items/*.jpg`).
Outputs saved to `../outputs/`.

In [ ]:
import os, sys, json
from pathlib import Path
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from PIL import Image

# Load .env from prototype root
load_dotenv(Path('..') / '.env')

# Add lib/ to path
sys.path.insert(0, str(Path('..')))

from lib.client import VISION_MODEL
from lib.images import resize_image
from lib.categorize import categorize_item
from lib.remove_background import remove_background, REMBG_AVAILABLE
from lib.metrics import MetricsRecorder

metrics = MetricsRecorder()
print(f'VISION_MODEL: {VISION_MODEL}')
print(f'API key set: {bool(os.getenv("GEMINI_API_KEY"))}')
print(f'rembg available: {REMBG_AVAILABLE}')
Path('../outputs').mkdir(exist_ok=True)

## Step 1 — Load input image

In [ ]:
# Pick input: DB item (preferred) or inputs/ photo
import glob

CLOSET_IMAGES = os.path.expanduser("~/.closet/images/items/*.jpg")
db_items = sorted(glob.glob(CLOSET_IMAGES))

if db_items:
    print(f"Found {len(db_items)} items in ~/.closet/images/items/")
    for i, p in enumerate(db_items[:10]):
        print(f"  [{i}] {os.path.basename(p)}")
    IMG_INDEX = 0  # ← change to pick a different item
    img_path = db_items[IMG_INDEX]
else:
    print("No DB items found — using inputs/ photo")
    img_path = Path("../inputs/20260524_174630.jpg")

print(f"
Using: {img_path}")
img_original = Image.open(img_path)
img = resize_image(img_original)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_original)
axes[0].set_title(f"Original ({img_original.width}×{img_original.height})")
axes[0].axis("off")
axes[1].imshow(img)
axes[1].set_title(f"Resized ({img.width}×{img.height})")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## Step 2 — Categorize with Gemini

Sends the image to Gemini and returns structured tags.

In [ ]:
print('Sending to Gemini for categorization...')
with metrics.record('gemini', 'none') as run:
    classification = categorize_item(img)
    run.items_found = 1
    run.cost_usd_est = 0.001  # rough estimate

print(json.dumps(classification, indent=2))

## Step 3 — Render data card

Mirrors the shape of `AddItem.tsx` step 3 confirm screen.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(img)
axes[0].set_title('Item photo')
axes[0].axis('off')

tags_text = '\n'.join([
    f"Category:    {classification.get('category', '—')}",
    f"Subcategory: {classification.get('subcategory', '—')}",
    f"Color:       {classification.get('primary_color', '—')}",
    f"Material:    {classification.get('material', '—')}",
    f"Season:      {', '.join(classification.get('season', []))}",
    f"Tags:        {', '.join(classification.get('suggested_tags', []))}",
    f"\nDescription:\n{classification.get('ai_description', '—')}",
])
axes[1].text(0.05, 0.95, tags_text, transform=axes[1].transAxes,
             fontsize=11, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
axes[1].axis('off')
axes[1].set_title('AI Classification')

plt.suptitle('Item Data Card', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/01_data_card.png', dpi=150)
plt.show()
print('Saved → outputs/01_data_card.png')

## Step 4 — Background removal (rembg)

> Requires `pip install rembg onnxruntime` if not installed. First run downloads ~175MB ONNX model.

In [ ]:
if not REMBG_AVAILABLE:
    print('rembg not installed. Run: pip install rembg onnxruntime')
else:
    print('Removing background with rembg...')
    with metrics.record('gemini', 'rembg') as run:
        cutout = remove_background(img, provider='rembg')
        run.items_found = 1

    # Composite on grey background for display
    display = Image.new('RGBA', cutout.size, (200, 200, 200, 255))
    if cutout.mode == 'RGBA':
        display.paste(cutout, mask=cutout.split()[3])
    else:
        display = cutout.convert('RGBA')

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(img)
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(display)
    axes[1].set_title('Cutout (rembg)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.savefig('../outputs/01_cutout.png', dpi=150)
    plt.show()
    print('Saved → outputs/01_cutout.png')

## Step 5 — Re-categorize the cutout

Does a clean transparent background improve Gemini's accuracy?

In [ ]:
if not REMBG_AVAILABLE:
    print('Skipping — rembg not installed')
else:
    print('Re-categorizing cutout...')
    classification_cutout = categorize_item(cutout.convert('RGB'))

    print('Original classification:')
    print(f"  category={classification.get('category')}  "
          f"subcategory={classification.get('subcategory')}  "
          f"color={classification.get('primary_color')}")
    print('Cutout classification:')
    print(f"  category={classification_cutout.get('category')}  "
          f"subcategory={classification_cutout.get('subcategory')}  "
          f"color={classification_cutout.get('primary_color')}")

## Buffer — Human Review

Inspect the proposed tags. Edit `classification` in the cell above if needed, then run the next cell to accept.

In [ ]:
print('=' * 60)
print('PROPOSED ITEM TAGS (edit classification dict above to change)')
print('=' * 60)
print(json.dumps(classification, indent=2))

## Metrics

In [ ]:
df = metrics.to_dataframe()
print(df.to_string(index=False))